# EXACT 2026 — Kaggle End-to-End Pipeline

Notebook này gom pipeline hiện tại vào **một file runnable trên Kaggle** thay vì import nhiều module local `src/exact/...`.

Default mode là `heuristic` để chạy nhanh và ổn định trên Kaggle CPU. Có thể bật LLM semantic parser bằng OpenAI-compatible API hoặc local HuggingFace model ở cell cấu hình.

Data flow:

```text
input JSON batch
  -> PredictionRequest
  -> TaskRouter
  -> Type1 logic: LLM/heuristic translation -> KB -> ForwardChainSolver -> PredictionResponse
  -> Type2 physics placeholder
  -> predictions.json
```


## 1. Optional dependency install

Kaggle thường có `torch`, `pandas` sẵn. Cell này cài các package cần cho pipeline. Nếu notebook restart kernel sau install thì chạy lại từ đầu.


In [ ]:
# Uncomment if Kaggle environment is missing dependencies.
# !pip install -q pydantic pydantic-settings openai transformers accelerate


## 2. Imports, configuration, and logging

Chỉnh các biến ở đây để chạy trên Kaggle.

- `LLM_PROVIDER = "none"`: dùng heuristic parser, nhanh nhất.
- `LLM_PROVIDER = "openai"`: gọi OpenAI-compatible API như Alibaba DashScope compatible mode.
- `LLM_PROVIDER = "local"`: load HuggingFace model local trên Kaggle GPU, chậm nếu model lớn.


In [ ]:
from __future__ import annotations

import asyncio
import hashlib
import importlib.util
import json
import logging
import os
import re
import sys
import time
import unicodedata
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Iterable, Literal, Protocol

from pydantic import BaseModel, ConfigDict, Field, SecretStr, field_validator

try:
    from openai import AsyncOpenAI
except Exception:
    AsyncOpenAI = None

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
except Exception:
    torch = None
    AutoModelForCausalLM = None
    AutoTokenizer = None

# ========= Kaggle/runtime config =========
INPUT_PATH = Path(os.getenv("EXACT_INPUT_PATH", "/kaggle/input/exact2026/Logic_Based_Educational_Queries_inference.json"))
OUTPUT_PATH = Path(os.getenv("EXACT_OUTPUT_PATH", "/kaggle/working/predictions.json"))
LIMIT = int(os.getenv("EXACT_LIMIT", "10"))  # set 0 for full dataset
PROGRESS_EVERY = int(os.getenv("EXACT_PROGRESS_EVERY", "1"))

LLM_PROVIDER = os.getenv("EXACT_LLM_PROVIDER", "none")  # none | openai | local
LLM_MODEL = os.getenv("EXACT_LLM_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
LLM_BASE_URL = os.getenv("EXACT_LLM_BASE_URL")
LLM_API_KEY = os.getenv("EXACT_LLM_API_KEY")
LLM_MAX_TOKENS = int(os.getenv("EXACT_MAX_NEW_TOKENS", "1024"))
LLM_TEMPERATURE = float(os.getenv("EXACT_LLM_TEMPERATURE", "0.0"))
LLM_TIMEOUT_SECONDS = float(os.getenv("EXACT_LLM_TIMEOUT_SECONDS", "60"))
LLM_MAX_RETRIES = int(os.getenv("EXACT_MAX_RETRIES", "2"))
REQUIRE_LLM = os.getenv("EXACT_REQUIRE_LLM", "0") == "1"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
)
logger = logging.getLogger("exact_kaggle")

logger.info(
    "config: input=%s output=%s limit=%s provider=%s model=%s require_llm=%s max_tokens=%s",
    INPUT_PATH,
    OUTPUT_PATH,
    LIMIT,
    LLM_PROVIDER,
    LLM_MODEL,
    REQUIRE_LLM,
    LLM_MAX_TOKENS,
)


## 3. Dataset schemas and task router

Copy/adapt từ `datasets/schemas.py` và `router/task_router.py`.


In [ ]:
class TaskType(str, Enum):
    TYPE1_LOGIC = "type1_logic"
    TYPE2_PHYSICS = "type2_physics"
    UNKNOWN = "unknown"


class QuestionType(str, Enum):
    MCQ = "mcq"
    YES_NO_UNCERTAIN = "yes_no_uncertain"
    OPEN_ENDED = "open_ended"
    NUMERICAL = "numerical"
    UNKNOWN = "unknown"


class AppBaseModel(BaseModel):
    model_config = ConfigDict(populate_by_name=True, extra="forbid", str_strip_whitespace=True)


class InboundBaseModel(BaseModel):
    model_config = ConfigDict(populate_by_name=True, extra="allow", str_strip_whitespace=True)


class PredictionRequest(InboundBaseModel):
    id: str | None = None
    question: str
    premises_nl: list[str] | None = Field(default=None, alias="premises-NL")

    @field_validator("question")
    @classmethod
    def question_must_not_be_empty(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("question must not be empty")
        return value

    @property
    def inferred_task_type(self) -> TaskType:
        if self.premises_nl:
            return TaskType.TYPE1_LOGIC
        return TaskType.TYPE2_PHYSICS


class PredictionResponse(AppBaseModel):
    answer: str
    explanation: str
    fol: str | None = None
    cot: list[str] | None = None
    premises: list[str] | None = None
    confidence: float | None = Field(default=None, ge=0.0, le=1.0)
    id: str | None = None
    task_type: TaskType | None = None
    question_type: QuestionType = QuestionType.UNKNOWN
    unit: str | None = None
    error: str | None = None


def to_official_response(response: PredictionResponse) -> dict[str, Any]:
    return {
        "answer": response.answer,
        "explanation": response.explanation,
        "fol": response.fol,
        "cot": response.cot,
        "premises": response.premises,
        "confidence": response.confidence,
    }


@dataclass(frozen=True)
class RouteDecision:
    task_type: TaskType
    reason: str


class TaskRouter:
    def route(self, request: PredictionRequest) -> RouteDecision:
        if request.premises_nl:
            return RouteDecision(TaskType.TYPE1_LOGIC, "premises_nl present")
        return RouteDecision(TaskType.TYPE2_PHYSICS, "premises_nl absent")


## 4. Logic IR and heuristic parser

Copy/adapt từ `logic/ir.py` và `logic/parser.py`.


In [ ]:
@dataclass(frozen=True, order=True)
class Atom:
    pred: str
    args: tuple[str, ...] = ()
    negated: bool = False
    text: str | None = None

    def positive(self) -> "Atom":
        return Atom(pred=self.pred, args=self.args, negated=False, text=self.text)

    def negation(self) -> "Atom":
        return Atom(pred=self.pred, args=self.args, negated=not self.negated, text=self.text)

    def display(self) -> str:
        label = self.text or (f"{self.pred}({', '.join(self.args)})" if self.args else self.pred.replace("_", " "))
        return f"not {label}" if self.negated else label


@dataclass(frozen=True)
class Rule:
    conditions: tuple[Atom, ...]
    conclusion: Atom
    source_idx: int
    text: str


@dataclass(frozen=True)
class Fact:
    atom: Atom
    source_idx: int
    text: str


@dataclass(frozen=True)
class ProofStep:
    derived: Atom
    used_premises: tuple[int, ...]
    rule_idx: int | None
    parents: tuple[Atom, ...] = ()
    natural_language: str | None = None


@dataclass(frozen=True)
class ParsedPremise:
    facts: tuple[Fact, ...] = ()
    rules: tuple[Rule, ...] = ()
    warnings: tuple[str, ...] = ()


@dataclass(frozen=True)
class Query:
    claim: Atom
    raw_question: str
    expects_negation: bool = False


@dataclass(frozen=True)
class SolveResult:
    label: str
    claim: Atom
    proof: tuple[ProofStep, ...] = ()
    supporting_premises: tuple[int, ...] = ()
    mode: str = "symbolic_forward_chain"
    warnings: tuple[str, ...] = ()


@dataclass(frozen=True)
class Theory:
    sorts: dict[str, list[str]] = field(default_factory=dict)
    predicates: dict[str, tuple[str, ...]] = field(default_factory=dict)
    functions: dict[str, tuple[tuple[str, ...], str]] = field(default_factory=dict)
    constants: dict[str, str] = field(default_factory=dict)


_IF_THEN_RE = re.compile(r"^\s*if\s+(.+?)\s*,?\s+then\s+(.+?)\.?\s*$", re.IGNORECASE)
_TRAILING_PUNCT_RE = re.compile(r"[\s.?!:;]+$")
_WORD_RE = re.compile(r"[a-z0-9]+")


def parse_premise_to_ir(premise: str, source_idx: int) -> ParsedPremise:
    text = premise.strip()
    if not text:
        return ParsedPremise(warnings=(f"premise {source_idx + 1} is empty",))

    match = _IF_THEN_RE.match(text)
    if match:
        antecedent, consequent = match.groups()
        conditions = tuple(_atom_from_clause(part) for part in _split_conjunction(antecedent))
        conclusion = _atom_from_clause(consequent)
        return ParsedPremise(rules=(Rule(conditions, conclusion, source_idx, text),))

    return ParsedPremise(facts=(Fact(_atom_from_clause(text), source_idx, text),))


def parse_question_to_query(question: str) -> Query:
    raw = question.strip()
    normalized = _strip_question_shell(raw)
    atom = _atom_from_clause(normalized)
    return Query(claim=atom, raw_question=raw, expects_negation=atom.negated)


def atom_from_text(text: str) -> Atom:
    return _atom_from_clause(text)


def _split_conjunction(text: str) -> list[str]:
    parts = re.split(r"\s+(?:and|&)\s+", text, flags=re.IGNORECASE)
    return [part.strip() for part in parts if part.strip()]


def _atom_from_clause(text: str) -> Atom:
    clause = _clean_clause(text)
    negated = False
    for prefix in ("it is not true that ", "not ", "does not ", "do not ", "did not "):
        if clause.startswith(prefix):
            negated = True
            clause = clause[len(prefix):].strip()
            break
    pred = _slugify(_canonicalize_clause(clause)) or "unknown"
    return Atom(pred=pred, negated=negated, text=clause)


def _strip_question_shell(question: str) -> str:
    text = _clean_clause(question)
    text = re.sub(r"^(?:based on the above premises,\s*)?(?:does|do|did|is|are|can|could|will|would|should)\s+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+(?:hold|holds|follow|follows)$", "", text, flags=re.IGNORECASE)
    return text.strip()


def _clean_clause(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.strip().strip("\"'")
    text = _TRAILING_PUNCT_RE.sub("", text)
    return re.sub(r"\s+", " ", text).lower()


def _canonicalize_clause(text: str) -> str:
    text = re.sub(r"^(?:a|an|the)\s+", "", text)
    text = re.sub(r"\s+(?:is|are|was|were)\s+true$", "", text)
    return text.strip()


def _slugify(text: str) -> str:
    return "_".join(_WORD_RE.findall(text))


## 5. Knowledge base, solver, and explanation

Copy/adapt từ `logic/kb.py`, `symbolic_solvers/forward_chain/solver.py`, và `logic/explain.py`.


In [ ]:
PARSER_VERSION = "heuristic_horn_v1"


@dataclass(frozen=True)
class KnowledgeBase:
    raw_premises: tuple[str, ...]
    facts: tuple[Fact, ...]
    rules: tuple[Rule, ...]
    premise_hash: str
    parser_version: str = PARSER_VERSION
    theory: Theory | None = None
    warnings: tuple[str, ...] = ()


def hash_premises(premises: list[str] | tuple[str, ...], parser_version: str = PARSER_VERSION) -> str:
    text = parser_version + "\n" + "\n".join(premises)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def build_kb_from_parsed_premises(
    premises: list[str] | tuple[str, ...],
    parsed_premises: tuple[ParsedPremise, ...],
    premise_hash: str | None = None,
    parser_version: str = PARSER_VERSION,
    extra_warnings: tuple[str, ...] = (),
) -> KnowledgeBase:
    facts: list[Fact] = []
    rules: list[Rule] = []
    warnings: list[str] = list(extra_warnings)
    raw_premises = tuple(premises)

    for parsed in parsed_premises:
        facts.extend(parsed.facts)
        rules.extend(parsed.rules)
        warnings.extend(parsed.warnings)

    return KnowledgeBase(
        raw_premises=raw_premises,
        facts=tuple(facts),
        rules=tuple(rules),
        premise_hash=premise_hash or hash_premises(raw_premises, parser_version),
        parser_version=parser_version,
        warnings=tuple(warnings),
    )


@dataclass(frozen=True)
class ForwardChainSolver:
    name: str = "forward_chain_horn"

    def solve(self, kb: KnowledgeBase, claim: Atom) -> SolveResult:
        return solve_query(kb, claim, mode=self.name)


def solve_query(kb: KnowledgeBase, claim: Atom, mode: str = "forward_chain_horn") -> SolveResult:
    known, proofs = derive_closure(kb)
    if claim in known:
        proof = _trace_proof(claim, proofs)
        return SolveResult("Yes", claim, tuple(proof), _support_from_proof(proof), mode, kb.warnings)

    negated_claim = claim.negation()
    if negated_claim in known:
        proof = _trace_proof(negated_claim, proofs)
        return SolveResult("No", claim, tuple(proof), _support_from_proof(proof), mode, kb.warnings)

    return SolveResult("Unknown", claim, (), (), mode, kb.warnings)


def derive_closure(kb: KnowledgeBase) -> tuple[set[Atom], dict[Atom, ProofStep]]:
    known: set[Atom] = set()
    proofs: dict[Atom, ProofStep] = {}

    for fact in kb.facts:
        if fact.atom not in known:
            known.add(fact.atom)
            proofs[fact.atom] = ProofStep(
                derived=fact.atom,
                used_premises=(fact.source_idx,),
                rule_idx=None,
                parents=(),
                natural_language=f"Premise {fact.source_idx + 1} states {fact.atom.display()}.",
            )

    changed = True
    while changed:
        changed = False
        for rule in kb.rules:
            if rule.conclusion in known:
                continue
            if all(condition in known for condition in rule.conditions):
                known.add(rule.conclusion)
                parent_premises: list[int] = [rule.source_idx]
                for condition in rule.conditions:
                    parent_premises.extend(proofs[condition].used_premises)
                proofs[rule.conclusion] = ProofStep(
                    derived=rule.conclusion,
                    used_premises=tuple(sorted(set(parent_premises))),
                    rule_idx=rule.source_idx,
                    parents=rule.conditions,
                    natural_language=(
                        f"Premise {rule.source_idx + 1} derives {rule.conclusion.display()} "
                        f"when {', '.join(parent.display() for parent in rule.conditions)} hold."
                    ),
                )
                changed = True

    return known, proofs


def _trace_proof(target: Atom, proofs: dict[Atom, ProofStep]) -> list[ProofStep]:
    ordered: list[ProofStep] = []
    visited: set[Atom] = set()

    def visit(atom: Atom) -> None:
        if atom in visited or atom not in proofs:
            return
        visited.add(atom)
        for parent in proofs[atom].parents:
            visit(parent)
        ordered.append(proofs[atom])

    visit(target)
    return ordered


def _support_from_proof(proof: list[ProofStep]) -> tuple[int, ...]:
    support: set[int] = set()
    for step in proof:
        support.update(step.used_premises)
    return tuple(sorted(support))


def explain_result(result: SolveResult, kb: KnowledgeBase) -> tuple[str, list[str], list[str]]:
    premise_labels = [f"P{idx + 1}" for idx in result.supporting_premises]
    if result.label == "Unknown":
        return (
            "The provided premises do not prove the claim or its negation, so the answer is Unknown.",
            ["No symbolic proof was found for the claim or for its negation."],
            [],
        )

    cot = [step.natural_language or f"Derived {step.derived.display()}." for step in result.proof]
    support_text = ", ".join(premise_labels) if premise_labels else "the parsed premises"
    if result.label == "Yes":
        explanation = f"Using {support_text}, the symbolic proof derives {result.claim.display()}."
    else:
        explanation = f"Using {support_text}, the symbolic proof derives the negation of {result.claim.display()}."
    return explanation, cot, premise_labels


def kb_to_fol_like_text(kb: KnowledgeBase) -> str:
    lines: list[str] = []
    for fact in kb.facts:
        lines.append(f"P{fact.source_idx + 1}: {fact.atom.display()}")
    for rule in kb.rules:
        conditions = " AND ".join(condition.display() for condition in rule.conditions)
        lines.append(f"P{rule.source_idx + 1}: {conditions} -> {rule.conclusion.display()}")
    return "\n".join(lines)


## 6. LLM clients and semantic translator

Copy/adapt từ `llm_client.py` và `logic/llm_translator.py`. Notebook vẫn chạy được khi `LLM_PROVIDER="none"`.


In [ ]:
class LLMClient:
    def __init__(self, api_key: str, base_url: str | None = None, model: str = "gpt-4o-mini", timeout: float = 60.0, max_retries: int = 2):
        if AsyncOpenAI is None:
            raise ImportError("openai package is required for LLM_PROVIDER='openai'")
        self.model = model
        self.client = AsyncOpenAI(api_key=api_key, base_url=base_url, timeout=timeout, max_retries=max_retries)

    async def complete_json(self, messages: Iterable[dict[str, Any]], temperature: float = 0.0, max_tokens: int = 2048) -> dict[str, Any]:
        response = await self.client.chat.completions.create(
            model=self.model,
            messages=list(messages),
            temperature=temperature,
            max_tokens=max_tokens,
            response_format={"type": "json_object"},
        )
        choice = response.choices[0]
        text = choice.message.content or ""
        try:
            return _parse_json_object(text)
        except ValueError as exc:
            raise ValueError(f"LLM returned invalid JSON with finish_reason={choice.finish_reason}: {text}") from exc

    def complete_json_sync(self, messages: Iterable[dict[str, Any]], temperature: float = 0.0, max_tokens: int = 2048) -> dict[str, Any]:
        try:
            asyncio.get_running_loop()
        except RuntimeError:
            return asyncio.run(self.complete_json(messages, temperature=temperature, max_tokens=max_tokens))
        raise RuntimeError("complete_json_sync cannot run inside an active event loop")


class LocalClient:
    def __init__(self, model_name: str):
        if AutoTokenizer is None or AutoModelForCausalLM is None or torch is None:
            raise ImportError("transformers and torch are required for LLM_PROVIDER='local'")
        logger.info("Loading local tokenizer for %s", model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        model_kwargs: dict[str, Any] = {"dtype": torch.float16 if torch.cuda.is_available() else torch.float32}
        if importlib.util.find_spec("accelerate") is not None:
            model_kwargs["device_map"] = "auto"
        logger.info("Loading local model %s with %s", model_name, model_kwargs)
        started_at = time.monotonic()
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        if "device_map" not in model_kwargs and torch.cuda.is_available():
            self.model = self.model.to("cuda")
        logger.info("Loaded local model in %.1fs", time.monotonic() - started_at)

    def complete(self, messages: list[dict[str, Any]], max_new_tokens: int = 2048) -> str:
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        input_length = inputs["input_ids"].shape[1]
        logger.info("Starting local generation: input_tokens=%s max_new_tokens=%s", input_length, max_new_tokens)
        started_at = time.monotonic()
        outputs = self.model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        generated_tokens = outputs[0][input_length:]
        logger.info("Finished local generation: output_tokens=%s elapsed=%.1fs", len(generated_tokens), time.monotonic() - started_at)
        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True)


class LocalJsonClient:
    def __init__(self, model_name: str):
        self.client = LocalClient(model_name)

    def complete_json_sync(self, messages: Iterable[dict[str, Any]], temperature: float = 0.0, max_tokens: int = 2048) -> dict[str, Any]:
        return _parse_json_object(self.client.complete(list(messages), max_new_tokens=max_tokens))


def build_json_client() -> Any | None:
    if LLM_PROVIDER == "local":
        return LocalJsonClient(LLM_MODEL)
    if LLM_PROVIDER == "openai":
        if not LLM_BASE_URL:
            raise ValueError("LLM_BASE_URL is required for LLM_PROVIDER='openai'")
        return LLMClient(
            api_key=LLM_API_KEY or "EMPTY",
            base_url=LLM_BASE_URL,
            model=LLM_MODEL,
            timeout=LLM_TIMEOUT_SECONDS,
            max_retries=LLM_MAX_RETRIES,
        )
    return None


def _parse_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1:
        raise ValueError(f"LLM output did not contain a JSON object: {text}")
    if end == -1 or end < start:
        raise ValueError(f"LLM output contained incomplete JSON; increase EXACT_MAX_NEW_TOKENS: {text}")
    try:
        parsed = json.loads(text[start:end + 1])
    except json.JSONDecodeError as exc:
        raise ValueError(f"LLM returned invalid JSON: {text}") from exc
    if not isinstance(parsed, dict):
        raise ValueError("LLM JSON output must be an object")
    return parsed


class AtomSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")
    text: str
    negated: bool = False

    @field_validator("text")
    @classmethod
    def text_must_not_be_empty(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("atom text must not be empty")
        return value


class RuleSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")
    conditions: list[AtomSpec] = Field(default_factory=list)
    conclusion: AtomSpec


class PremiseSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")
    source_idx: int
    facts: list[AtomSpec] = Field(default_factory=list)
    rules: list[RuleSpec] = Field(default_factory=list)


class QuerySpec(BaseModel):
    model_config = ConfigDict(extra="forbid")
    claim: AtomSpec


class TranslationSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")
    premises: list[PremiseSpec]
    query: QuerySpec


def translate_with_llm(premises: list[str], question: str, llm_client: Any) -> tuple[tuple[ParsedPremise, ...], Query]:
    messages = _build_messages(premises, question)
    logger.info("Starting LLM translation: premises=%s question_chars=%s max_tokens=%s", len(premises), len(question), LLM_MAX_TOKENS)
    raw = llm_client.complete_json_sync(messages=messages, temperature=LLM_TEMPERATURE, max_tokens=LLM_MAX_TOKENS)
    spec = TranslationSpec.model_validate(raw)
    return _spec_to_ir(spec, premises, question)


def translate_with_fallback(premises: list[str], question: str, llm_client: Any | None, allow_heuristic_fallback: bool = True) -> tuple[tuple[ParsedPremise, ...], Query, tuple[str, ...]]:
    if llm_client is not None:
        try:
            parsed, query = translate_with_llm(premises, question, llm_client)
            return parsed, query, ()
        except Exception as exc:
            if not allow_heuristic_fallback:
                raise RuntimeError(f"LLM translation failed and fallback is disabled: {exc}") from exc
            warnings = (f"LLM translation failed; heuristic parser used: {exc}",)
            return _heuristic_translation(premises, question, warnings)

    if not allow_heuristic_fallback:
        raise RuntimeError("No LLM client configured and fallback is disabled")
    return _heuristic_translation(premises, question, ("No LLM client configured; heuristic parser used.",))


def _heuristic_translation(premises: list[str], question: str, warnings: tuple[str, ...] = ()) -> tuple[tuple[ParsedPremise, ...], Query, tuple[str, ...]]:
    parsed = tuple(parse_premise_to_ir(premise, idx) for idx, premise in enumerate(premises))
    query = parse_question_to_query(question)
    return parsed, query, warnings


def _build_messages(premises: list[str], question: str) -> list[dict[str, str]]:
    premise_text = "\n".join(f"{idx}: {premise}" for idx, premise in enumerate(premises))
    return [
        {
            "role": "system",
            "content": (
                "You are a semantic parser for an educational logic QA system. "
                "Translate natural-language premises and the question into a compact Horn-style JSON IR. "
                "Do not answer the question. Use only the given text."
            ),
        },
        {
            "role": "user",
            "content": (
                "Return exactly this JSON shape:\n"
                "{\n"
                '  "premises": [\n'
                '    {"source_idx": 0, "facts": [{"text": "A", "negated": false}], '
                '"rules": [{"conditions": [{"text": "A"}], "conclusion": {"text": "B"}}]}\n'
                "  ],\n"
                '  "query": {"claim": {"text": "B", "negated": false}}\n'
                "}\n\n"
                "Rules:\n"
                "- source_idx must match the premise number shown below.\n"
                "- Use facts for directly stated atomic statements.\n"
                "- Use rules for if/then, who/that/when conditional rules, requirements, implications.\n"
                "- Split conjunctions into multiple condition atoms.\n"
                "- Preserve entities and predicates in simple English text.\n"
                "- Mark negated=true only for explicit negation.\n\n"
                f"Premises:\n{premise_text}\n\nQuestion:\n{question}"
            ),
        },
    ]


def _spec_to_ir(spec: TranslationSpec, raw_premises: list[str], raw_question: str) -> tuple[tuple[ParsedPremise, ...], Query]:
    parsed_by_idx: dict[int, ParsedPremise] = {}
    for premise_spec in spec.premises:
        source_idx = premise_spec.source_idx
        if source_idx < 0 or source_idx >= len(raw_premises):
            continue
        facts = [Fact(_atom_from_spec(atom_spec), source_idx, raw_premises[source_idx]) for atom_spec in premise_spec.facts]
        rules = []
        warnings = []
        for rule_spec in premise_spec.rules:
            conclusion = _atom_from_spec(rule_spec.conclusion)
            if not rule_spec.conditions:
                facts.append(Fact(conclusion, source_idx, raw_premises[source_idx]))
                warnings.append(f"LLM emitted rule with empty conditions for premise {source_idx}; treated conclusion as fact.")
                continue
            rules.append(
                Rule(
                    tuple(_atom_from_spec(atom_spec) for atom_spec in rule_spec.conditions),
                    conclusion,
                    source_idx,
                    raw_premises[source_idx],
                )
            )
        parsed_by_idx[source_idx] = ParsedPremise(facts=tuple(facts), rules=tuple(rules), warnings=tuple(warnings))

    parsed = []
    for source_idx, _premise in enumerate(raw_premises):
        parsed.append(parsed_by_idx.get(source_idx) or ParsedPremise(warnings=(f"No LLM IR for premise {source_idx}",)))
    return tuple(parsed), Query(_atom_from_spec(spec.query.claim), raw_question)


def _atom_from_spec(spec: AtomSpec) -> Atom:
    atom = atom_from_text(spec.text)
    return Atom(pred=atom.pred, args=atom.args, negated=spec.negated, text=atom.text)


## 7. Type-specific pipelines and batch runner

Copy/adapt từ `logic/pipeline.py`, `type2/pipeline.py`, và `scripts/run_predictions.py`.


In [ ]:
TYPE2_NOT_IMPLEMENTED_MESSAGE = (
    "Type 2 physics reasoning is reserved for the dedicated physics pipeline. "
    "This placeholder keeps the API contract stable while the team implements "
    "paper-backed quantity extraction, formula selection, execution, and verification."
)


def run_type1_pipeline(request: PredictionRequest, translator_client: Any | None = None, allow_heuristic_fallback: bool = True) -> PredictionResponse:
    logger.info("Start Type 1 pipeline id=%s", request.id)
    premises = request.premises_nl or []
    parsed_premises, query, translation_warnings = translate_with_fallback(
        premises=premises,
        question=request.question,
        llm_client=translator_client,
        allow_heuristic_fallback=allow_heuristic_fallback,
    )
    parser_version = "llm_translator_v1" if translator_client is not None else "heuristic_horn_v1"
    kb = build_kb_from_parsed_premises(premises, parsed_premises, parser_version=parser_version, extra_warnings=translation_warnings)
    result = ForwardChainSolver().solve(kb, query.claim)
    explanation, cot, cited_premises = explain_result(result, kb)
    confidence = {"Yes": 0.78, "No": 0.76, "Unknown": 0.35}[result.label]
    return PredictionResponse(
        id=request.id,
        task_type=TaskType.TYPE1_LOGIC,
        question_type=QuestionType.YES_NO_UNCERTAIN,
        answer=result.label,
        explanation=explanation,
        fol=kb_to_fol_like_text(kb) or None,
        cot=cot,
        premises=cited_premises,
        confidence=confidence,
        error="; ".join(result.warnings) if result.warnings else None,
    )


def run_type2_pipeline(request: PredictionRequest) -> PredictionResponse:
    logger.info("Start Type 2 pipeline id=%s", request.id)
    return PredictionResponse(
        id=request.id,
        task_type=TaskType.TYPE2_PHYSICS,
        question_type=QuestionType.NUMERICAL,
        answer="",
        explanation=TYPE2_NOT_IMPLEMENTED_MESSAGE,
        fol=None,
        cot=[
            "The request was routed to the Type 2 physics branch.",
            "The production physics solver has not been implemented in this repo yet.",
        ],
        premises=[
            "Type 2 receives only the question text.",
            "A future physics pipeline should derive answers from formulas, units, and verified computation.",
        ],
        confidence=0.0,
        error="type2_pipeline_not_implemented",
    )


def load_instances(path: Path) -> list[dict[str, Any]]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    records = _extract_records(payload, path)
    instances: list[dict[str, Any]] = []

    for group_index, record in enumerate(records):
        if not isinstance(record, dict):
            raise ValueError(f"Expected object at index {group_index} in {path}, got {type(record).__name__}")
        instances.extend(_record_to_prediction_instances(record, group_index))

    if not instances:
        raise ValueError(f"No prediction instances found in {path}")
    return instances


def _extract_records(payload: Any, path: Path) -> list[Any]:
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and isinstance(payload.get("instances"), list):
        return payload["instances"]
    if isinstance(payload, dict) and isinstance(payload.get("data"), list):
        return payload["data"]
    if isinstance(payload, dict):
        return [payload]
    raise ValueError(f"Unsupported JSON structure in {path}: {type(payload).__name__}")


def _record_to_prediction_instances(record: dict[str, Any], group_index: int) -> list[dict[str, Any]]:
    if record.get("question") is not None:
        return [_normalize_prediction_record(record, f"record_{group_index:04d}")]

    questions = record.get("questions")
    if isinstance(questions, list):
        premises = _normalize_premises(record.get("premises-NL") or record.get("premises_nl") or record.get("premises") or [])
        group_id = str(record.get("id") or record.get("group_id") or f"logic_{group_index:04d}")
        return [
            {
                "id": f"{group_id}_{question_index:02d}",
                "group_id": group_id,
                "question_index": question_index,
                "premises-NL": premises,
                "question": str(question).strip(),
            }
            for question_index, question in enumerate(questions)
            if str(question).strip()
        ]

    return [_normalize_prediction_record(record, f"record_{group_index:04d}")]


def _normalize_prediction_record(record: dict[str, Any], fallback_id: str) -> dict[str, Any]:
    question = record.get("question") or record.get("query") or record.get("problem")
    if question is None:
        raise ValueError(f"Record {fallback_id} has no question/query/problem field: keys={sorted(record)}")

    instance = {
        "id": record.get("id") or record.get("qid") or record.get("uid") or fallback_id,
        "question": str(question).strip(),
    }
    premises = record.get("premises-NL") or record.get("premises_nl") or record.get("premises") or record.get("context")
    if premises is not None:
        instance["premises-NL"] = _normalize_premises(premises)
    return instance


def _normalize_premises(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value]
    if isinstance(value, str):
        return [value]
    if value is None:
        return []
    raise ValueError(f"Invalid premises format: expected list[str] or str, got {type(value).__name__}")


def run_predictions(instances: list[dict[str, Any]], output_path: Path, limit: int = 0) -> dict[str, Any]:
    if limit:
        instances = instances[:limit]

    router = TaskRouter()
    translator_client = build_json_client()
    logger.info(
        "prediction runner: provider=%s model=%s llm_enabled=%s require_llm=%s instances=%s",
        LLM_PROVIDER,
        LLM_MODEL,
        translator_client is not None,
        REQUIRE_LLM,
        len(instances),
    )

    predictions: list[dict[str, Any]] = []
    for index, instance in enumerate(instances, start=1):
        request = PredictionRequest.model_validate(instance)
        route = router.route(request)
        logger.info("processing %s/%s id=%s route=%s", index, len(instances), request.id, route.task_type.value)

        if route.task_type == TaskType.TYPE1_LOGIC:
            response = run_type1_pipeline(
                request,
                translator_client=translator_client,
                allow_heuristic_fallback=not REQUIRE_LLM,
            )
        elif route.task_type == TaskType.TYPE2_PHYSICS:
            response = run_type2_pipeline(request)
        else:
            raise ValueError(f"Unsupported task type: {route.task_type}")

        prediction = response.model_dump(mode="json")
        prediction["route_reason"] = route.reason
        prediction["official"] = to_official_response(response)
        predictions.append(prediction)

        if PROGRESS_EVERY and index % PROGRESS_EVERY == 0:
            logger.info("processed %s/%s answer=%s", index, len(instances), response.answer)

    output = {
        "source": str(INPUT_PATH),
        "count": len(predictions),
        "format": "exact_predictions",
        "predictions": predictions,
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(output, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    logger.info("wrote %s predictions to %s", len(predictions), output_path)
    return output


## 8. Load Kaggle input data

Update `INPUT_PATH` ở cell config nếu dataset nằm ở path khác trong `/kaggle/input/...`. Nếu file không tồn tại, cell tạo một sample nhỏ để smoke test pipeline.


In [ ]:
if INPUT_PATH.exists():
    instances = load_instances(INPUT_PATH)
    logger.info("loaded %s instances from %s", len(instances), INPUT_PATH)
else:
    logger.warning("input path does not exist: %s; using a tiny smoke-test sample", INPUT_PATH)
    instances = [
        {
            "id": "sample_logic_001",
            "premises-NL": [
                "If a student studies, then the student passes.",
                "A student studies.",
            ],
            "question": "Does it follow that the student passes?",
        }
    ]

instances[0]


## 9. Run end-to-end predictions

Default `LIMIT=10`. Set environment variable `EXACT_LIMIT=0` before running notebook for full dataset, or edit the config cell.


In [ ]:
output = run_predictions(instances, OUTPUT_PATH, limit=LIMIT)
output["count"], OUTPUT_PATH


## 10. Inspect and download output

Kaggle output file is written to `/kaggle/working/predictions.json` by default.


In [ ]:
print(json.dumps(output["predictions"][:2], ensure_ascii=False, indent=2)[:4000])
print(f"Saved to: {OUTPUT_PATH}")
